# 04 · Migración a MongoDB Atlas

**Colegio Universitario de Cartago** · Minería de texto · Código BD-163

**Integrantes:** Marco Álvarez, Sharon Obando, Fabian Brenes
**Profesor:** Osvaldo Gonzalez Chaves · 2026

---

Este notebook documenta la creación de un entorno en **MongoDB Atlas** y su conexión con **Python**.
Cubre el registro de la cuenta, la organización, el proyecto **Movies**, el clúster **Cluster0**, la conexión y la migración de las reseñas procesadas a la base de datos.

## 1. Creación del entorno en MongoDB Atlas

1. **Cuenta:** crear una cuenta gratuita en [MongoDB Atlas](https://www.mongodb.com/products/platform/atlas-database).
2. **Organización:** es el nivel más alto de administración. Agrupa proyectos, usuarios, permisos y recursos. Se crea con un nombre alusivo a la solución.
   - En **Members and security** se comparte la organización con otras personas, usando su correo.
3. **Proyecto `Movies`:** es el contenedor de los recursos de una aplicación o entorno.
   - Para compartir solo el proyecto: **Add Members and Set Permissions**.
4. **Clúster `Cluster0`:** es el servidor (o conjunto de servidores) donde se almacenan y procesan las bases de datos. Se usa la **versión gratuita**, que tiene algunas limitaciones.
   - Al crearlo, Atlas genera las **credenciales** del usuario de base de datos.
5. **Método de conexión:** *Python → PyMongo (Synchronous)*. Atlas entrega el string de conexión:

```
mongodb+srv://<db_username>:<db_password>@cluster0.curaqng.mongodb.net/
```

**Jerarquía de MongoDB Atlas**
```
Organización
 └── Proyecto (Movies)
      └── Cluster (Cluster0)
           └── Base de datos (Movies)
                └── Colección (Movies_data)
                     └── Documentos
```

## 2. Instalación

In [ ]:
%pip install -q pymongo "pymongo[srv]" python-dotenv pandas

## 3. Credenciales

El usuario y la contraseña **no se escriben en el código**: se guardan en `atlas-credentials.env`, en la raíz del proyecto, y se agrega ese archivo a `.gitignore`.

```
MONGO_URI=mongodb+srv://<db_username>:<db_password>@cluster0.curaqng.mongodb.net/?retryWrites=true&w=majority
MONGO_DB=Movies
MONGO_COLLECTION=Movies_data
```

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Raíz del proyecto (el notebook vive en notebooks/ o en la raíz)
RAIZ = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()
RUTA_ENV = RAIZ / "atlas-credentials.env"

load_dotenv(RUTA_ENV)
MONGO_URI = os.getenv("MONGO_URI")
MONGO_DB = os.getenv("MONGO_DB", "Movies")
MONGO_COLLECTION = os.getenv("MONGO_COLLECTION", "Movies_data")

assert MONGO_URI, f"Falta MONGO_URI en {RUTA_ENV}"
print(f"Base de datos: {MONGO_DB} | Colección: {MONGO_COLLECTION}")

## 4. Prueba de conexión

`ping` comprueba que el clúster responde. Si todo está bien, debe mostrar **"Connected successfully"**.

In [ ]:
from pymongo import MongoClient

client = None
try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    print("Connected successfully")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")
finally:
    if client:
        client.close()

## 5. Base de datos y colección

**Conceptos**
| Concepto | Descripción |
|---|---|
| **Database** | Contenedor que agrupa varias colecciones relacionadas |
| **Collection** | Equivalente a una tabla en SQL |
| **Document** | Un registro en formato JSON (BSON); equivale a una fila |

**Creación en Atlas:** *Cluster0 → Browse collections → Data Explorer → Create Database*
- Database Name: `Movies`
- Collection Name: `Movies_data`

En la pestaña **Schema** se revisa la estructura, y con **Add Data → Insert Document** se pueden agregar documentos de forma manual.
MongoDB también crea la base de datos y la colección de forma automática con la primera inserción desde Python.

In [ ]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
db = client[MONGO_DB]
coleccion = db[MONGO_COLLECTION]

print("Bases de datos:", client.list_database_names())
print(f"Colecciones en {MONGO_DB}:", db.list_collection_names())

## 6. Insertar datos desde Python (migración)

Se migra `data/results/comparacion_spacy_nltk.csv` (las reseñas con el análisis de NLTK y spaCy) con la clase `MongoResenas` de `src/MongoDB_importador.py`.

| Paso | Detalle |
|---|---|
| Tipos | `rating` → número · `Fecha_comentario` → entero · `coincidencia_tokens` → booleano · tokens, lemas y POS → listas · entidades → `[{texto, tipo}]` |
| `_id` | Determinista (título + año + reseña + fuente): re-ejecutar **no duplica** documentos |
| Escritura | `ReplaceOne(..., upsert=True)` en lotes de 1000 |

In [ ]:
sys.path.insert(0, str(RAIZ / "src"))
from MongoDB_importador import MongoResenas, CSV_ENTRADA

# MONGO_DB y MONGO_COLLECTION se toman de atlas-credentials.env
with MongoResenas(ruta_env=RUTA_ENV, ruta_csv=CSV_ENTRADA) as importador:
    resultado = importador.importar(lote=1000)

resultado

## 7. Consultar datos desde Python

In [ ]:
import pandas as pd

print("Documentos en la colección:", coleccion.count_documents({}))

campos = {
    "_id": 0, "primaryTitle": 1, "review_summary_es": 1, "rating": 1, "valoracion": 1,
    "entidades_nltk": 1, "organizaciones_nltk": 1,
    "entidades_spacy": 1, "organizaciones_spacy": 1, "entidades_en_comun": 1,
}
df = pd.DataFrame(coleccion.find({}, campos).limit(20))
df.head()

In [ ]:
# Conteo por valoración y por tipo de obra
pd.DataFrame(coleccion.aggregate([
    {"$group": {"_id": {"valoracion": "$valoracion", "tipo": "$Tipo obra"}, "total": {"$sum": 1}}},
    {"$project": {"_id": 0, "valoracion": "$_id.valoracion", "tipo_obra": "$_id.tipo", "total": 1}},
    {"$sort": {"total": -1}},
]))

In [ ]:
# Entidades por tipo: NLTK vs spaCy
def entidades_por_tipo(campo):
    return {d["_id"]: d["total"] for d in coleccion.aggregate([
        {"$unwind": f"${campo}"},
        {"$group": {"_id": f"${campo}.tipo", "total": {"$sum": 1}}},
    ])}

pd.DataFrame({"NLTK": entidades_por_tipo("entidades_nltk"),
              "spaCy": entidades_por_tipo("entidades_spacy")}).fillna(0).astype(int)

In [ ]:
client.close()

## 8. Consultar datos desde MongoDB Atlas

En *Data Explorer → Movies → Movies_data → Documents*, se escribe el filtro en la barra **Filter**:

| Consulta | Filtro |
|---|---|
| Reseñas positivas | `{ valoracion: "positiva" }` |
| Series con rating ≥ 7 | `{ "Tipo obra": "Serie", rating: { $gte: 7 } }` |
| Reseñas con lugares según spaCy | `{ "entidades_spacy.tipo": "LOC" }` |
| Con entidades en común | `{ "entidades_en_comun.0": { $exists: true } }` |

En la pestaña **Aggregations** se pueden ejecutar los mismos pipelines de la sección 7.